# Mathlib Atlas — Data Exploration

Basic sanity checks and statistics on the ingested DuckDB database.

In [1]:
import sys
from pathlib import Path

import duckdb
import polars as pl

ROOT = Path().resolve().parent
DB_PATH = ROOT / "data" / "mathlib.db"

db = duckdb.connect(str(DB_PATH), read_only=True)
print(f"Opened: {DB_PATH}")

Opened: /home/matthew/mathlib-atlas/data/mathlib.db


## 1. Row Counts

In [2]:
db.sql("SELECT COUNT(*) AS total_declarations FROM declarations").show()
db.sql("SELECT COUNT(*) AS total_edges FROM dependencies").show()

┌────────────────────┐
│ total_declarations │
│       int64        │
├────────────────────┤
│             738240 │
└────────────────────┘

┌─────────────┐
│ total_edges │
│    int64    │
├─────────────┤
│      387509 │
└─────────────┘



## 2. Dataset Overlap

In [3]:
db.sql("""
SELECT
    SUM(CASE WHEN in_mltypes AND in_leandojo THEN 1 ELSE 0 END)  AS both,
    SUM(CASE WHEN in_mltypes AND NOT in_leandojo THEN 1 ELSE 0 END) AS only_mltypes,
    SUM(CASE WHEN in_leandojo AND NOT in_mltypes THEN 1 ELSE 0 END) AS only_leandojo,
    COUNT(*) AS total
FROM declarations
""").show()

┌────────┬──────────────┬───────────────┬────────┐
│  both  │ only_mltypes │ only_leandojo │ total  │
│ int128 │    int128    │    int128     │ int64  │
├────────┼──────────────┼───────────────┼────────┤
│ 160838 │       552277 │         25125 │ 738240 │
└────────┴──────────────┴───────────────┴────────┘



## 3. Kind Distribution

In [4]:
db.sql("""
SELECT kind, COUNT(*) AS n
FROM declarations
GROUP BY kind
ORDER BY n DESC
""").show()

┌──────────┬────────┐
│   kind   │   n    │
│ varchar  │ int64  │
├──────────┼────────┤
│ unknown  │ 519120 │
│ theorem  │ 143572 │
│ def      │  42331 │
│ instance │  16824 │
│ lemma    │  16383 │
│ axiom    │     10 │
└──────────┴────────┘



## 4. Module Breakdown (Top-Level)

In [5]:
db.sql("""
SELECT
    module_parts[1] AS top_level,
    COUNT(*) AS n
FROM declarations
WHERE module_parts IS NOT NULL AND len(module_parts) >= 1
GROUP BY 1
ORDER BY 2 DESC
LIMIT 15
""").show()

┌──────────────────┬────────┐
│    top_level     │   n    │
│     varchar      │ int64  │
├──────────────────┼────────┤
│ Mathlib          │ 524656 │
│ Lean             │  83358 │
│ Init             │  60464 │
│ Std              │  44619 │
│ Batteries        │   9479 │
│                  │   6607 │
│ Aesop            │   6584 │
│ ProofWidgets     │    967 │
│ Qq               │    613 │
│ Plausible        │    589 │
│ LeanSearchClient │    202 │
│ ImportGraph      │     96 │
│ Lake             │      6 │
├──────────────────┴────────┤
│ 13 rows         2 columns │
└───────────────────────────┘



In [6]:
# Second-level breakdown within Mathlib
db.sql("""
SELECT
    module_parts[2] AS area,
    COUNT(*) AS n
FROM declarations
WHERE module_parts[1] = 'Mathlib'
  AND len(module_parts) >= 2
GROUP BY 1
ORDER BY 2 DESC
LIMIT 20
""").show()

┌───────────────────┬───────┐
│       area        │   n   │
│      varchar      │ int64 │
├───────────────────┼───────┤
│ Algebra           │ 92847 │
│ CategoryTheory    │ 68706 │
│ Analysis          │ 46451 │
│ Topology          │ 44818 │
│ Data              │ 44604 │
│ RingTheory        │ 33732 │
│ Order             │ 31451 │
│ LinearAlgebra     │ 20918 │
│ MeasureTheory     │ 18902 │
│ GroupTheory       │ 16556 │
│ Tactic            │ 14639 │
│ Combinatorics     │ 13874 │
│ NumberTheory      │ 12131 │
│ AlgebraicGeometry │  9842 │
│ Geometry          │  8974 │
│ AlgebraicTopology │  6653 │
│ SetTheory         │  6351 │
│ Probability       │  6248 │
│ FieldTheory       │  4996 │
│ Computability     │  4758 │
├───────────────────┴───────┤
│ 20 rows         2 columns │
└───────────────────────────┘



## 5. Degree Distributions

In [7]:
# Out-degree (how many things does a theorem depend on?)
out_deg = db.sql("""
SELECT src AS name, COUNT(*) AS out_degree
FROM dependencies
GROUP BY src
ORDER BY out_degree DESC
LIMIT 20
""").pl()
print("Top 20 by out-degree (most premises used):")
print(out_deg)

Top 20 by out-degree (most premises used):
shape: (20, 2)
┌─────────────────────────────────┬────────────┐
│ name                            ┆ out_degree │
│ ---                             ┆ ---        │
│ str                             ┆ i64        │
╞═════════════════════════════════╪════════════╡
│ Besicovitch.exist_finset_disjo… ┆ 108        │
│ PhragmenLindelof.horizontal_st… ┆ 102        │
│ mem_adjoin_of_smul_prime_smul_… ┆ 101        │
│ MeasureTheory.hasFDerivAt_conv… ┆ 99         │
│ Vitali.exists_disjoint_coverin… ┆ 98         │
│ …                               ┆ …          │
│ NumberField.mixedEmbedding.con… ┆ 82         │
│ ContinuousMap.idealOfSet_ofIde… ┆ 82         │
│ AlgebraicTopology.DoldKan.σ_co… ┆ 82         │
│ IsCyclotomicExtension.discr_pr… ┆ 82         │
│ MeasureTheory.lintegral_abs_de… ┆ 81         │
└─────────────────────────────────┴────────────┘


In [8]:
# In-degree (how many theorems depend on this one?)
in_deg = db.sql("""
SELECT dst AS name, COUNT(*) AS in_degree
FROM dependencies
GROUP BY dst
ORDER BY in_degree DESC
LIMIT 20
""").pl()
print("Top 20 by in-degree (most widely used as premise):")
print(in_deg)

Top 20 by in-degree (most widely used as premise):
shape: (20, 2)
┌───────────────────────────────┬───────────┐
│ name                          ┆ in_degree │
│ ---                           ┆ ---       │
│ str                           ┆ i64       │
╞═══════════════════════════════╪═══════════╡
│ rfl                           ┆ 2645      │
│ Eq.symm                       ┆ 1730      │
│ mul_comm                      ┆ 1462      │
│ Iff.rfl                       ┆ 1413      │
│ Function.Injective            ┆ 1268      │
│ …                             ┆ …         │
│ eq_comm                       ┆ 701       │
│ LE.le.trans                   ┆ 699       │
│ CategoryTheory.Category.assoc ┆ 695       │
│ Set.univ                      ┆ 692       │
│ congr_arg                     ┆ 680       │
└───────────────────────────────┴───────────┘


In [9]:
# Degree distribution summary
db.sql("""
WITH deg AS (
    SELECT dst AS name, COUNT(*) AS in_degree FROM dependencies GROUP BY dst
)
SELECT
    MIN(in_degree)  AS min_in,
    MAX(in_degree)  AS max_in,
    AVG(in_degree)  AS mean_in,
    MEDIAN(in_degree) AS median_in,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY in_degree) AS p99_in
FROM deg
""").show()

┌────────┬────────┬───────────────────┬───────────┬────────┐
│ min_in │ max_in │      mean_in      │ median_in │ p99_in │
│ int64  │ int64  │      double       │  double   │ double │
├────────┼────────┼───────────────────┼───────────┼────────┤
│      1 │   2645 │ 5.003279492840635 │       2.0 │   58.0 │
└────────┴────────┴───────────────────┴───────────┴────────┘



## 6. Spot-Checks

In [10]:
spot_names = ["Nat.add_comm", "Finset.sum_comm", "Real.sqrt_sq", "List.length_append"]

for name in spot_names:
    row = db.execute(
        "SELECT name, module, kind, in_mltypes, in_leandojo FROM declarations WHERE name = ?",
        [name]
    ).fetchone()
    if row:
        print(f"  {row}")
    else:
        print(f"  NOT FOUND: {name}")

  ('Nat.add_comm', 'Init.Data.Nat.Basic', 'theorem', True, True)
  ('Finset.sum_comm', 'Mathlib.Algebra.BigOperators.Group.Finset.Sigma', 'unknown', True, True)
  ('Real.sqrt_sq', 'Mathlib.Data.Real.Sqrt', 'theorem', True, True)
  ('List.length_append', 'Init.Data.List.Basic', 'theorem', True, True)


In [11]:
# Dependencies of Nat.add_comm
db.sql("""
SELECT dst, d.kind, d.module
FROM dependencies dep
JOIN declarations d ON dep.dst = d.name
WHERE dep.src = 'Nat.add_comm'
""").show()

┌─────────┬─────────┬─────────┐
│   dst   │  kind   │ module  │
│ varchar │ varchar │ varchar │
├─────────┴─────────┴─────────┤
│           0 rows            │
└─────────────────────────────┘



In [12]:
# What depends on Nat.add_comm?
db.sql("""
SELECT src, d.kind, d.module
FROM dependencies dep
JOIN declarations d ON dep.src = d.name
WHERE dep.dst = 'Nat.add_comm'
LIMIT 20
""").show()

┌─────────────────────────────────────┬─────────┬──────────────────────────────────────┐
│                 src                 │  kind   │                module                │
│               varchar               │ varchar │               varchar                │
├─────────────────────────────────────┼─────────┼──────────────────────────────────────┤
│ Nat.multichoose_eq                  │ theorem │ Mathlib.Data.Nat.Choose.Basic        │
│ List.length_foldr_permutationsAux2' │ theorem │ Mathlib.Data.List.Permutation        │
│ BitVec.cons                         │ def     │ Init.Data.BitVec.Basic               │
│ List.length_eq_countP_add_countP    │ theorem │ Init.Data.List.Count                 │
│ Nat.testBit_bit_succ                │ lemma   │ Mathlib.Data.Nat.Bits                │
│ RelSeries.append                    │ def     │ Mathlib.Order.RelSeries              │
│ Nat.shiftLeft'_sub                  │ lemma   │ Mathlib.Data.Nat.Bits                │
│ Fin.append_rev     

## 7. Verification Queries (from plan)

In [13]:
print("Total declarations:")
db.sql("SELECT COUNT(*) FROM declarations").show()

print("Total edges:")
db.sql("SELECT COUNT(*) FROM dependencies").show()

print("In both datasets:")
db.sql("SELECT COUNT(*) FROM declarations WHERE in_leandojo AND in_mltypes").show()

print("Top-level module counts:")
db.sql("""
    SELECT module_parts[1], COUNT(*)
    FROM declarations
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").show()

Total declarations:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       738240 │
└──────────────┘

Total edges:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       387509 │
└──────────────┘

In both datasets:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       160838 │
└──────────────┘

Top-level module counts:
┌─────────────────┬──────────────┐
│ module_parts[1] │ count_star() │
│     varchar     │    int64     │
├─────────────────┼──────────────┤
│ Mathlib         │       524656 │
│ Lean            │        83358 │
│ Init            │        60464 │
│ Std             │        44619 │
│ Batteries       │         9479 │
│                 │         6607 │
│ Aesop           │         6584 │
│ ProofWidgets    │          967 │
│ Qq              │          613 │
│ Plausible       │          589 │
├─────────────────┴──────────────┤
│ 10 rows              2 columns │
└────────────────────────────────┘

